In [4]:
# =========================
# IMPORTS & SETUP
# =========================
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from torch.utils.data import WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import os
from PIL import Image
import numpy as np
from torch.amp import GradScaler, autocast

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================
# CLASS MAPPING & AUGMENTATIONS
# =========================
class_mapping = {
    "Corn_Gray_leaf_spot": 0, "Corn_leaf_blight": 1, "Corn_rust_leaf": 2,
    "Tomato_Septoria_leaf_spot": 3, "Tomato_leaf": 4, "Apple_Scab_Leaf": 5,
    "Apple_leaf": 6, "Apple_rust_leaf": 7, "grape_leaf": 8, "grape_leaf_black_rot": 9,
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": 0,
    "Corn_(maize)___Northern_Leaf_Blight": 1, "Corn_(maize)___Common_rust_": 2,
    "Tomato___Septoria_leaf_spot": 3, "Tomato___healthy": 4,
    "Apple___Apple_scab": 5, "Apple___healthy": 6, "Apple___Cedar_apple_rust": 7,
    "Grape___healthy": 8, "Grape___Black_rot": 9,
}

train_transform = transforms.Compose([
    transforms.Resize((400, 400)),
    transforms.RandomResizedCrop(384, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((400, 400)),
    transforms.CenterCrop(384),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# =========================
# DATASETS & DATALOADERS
# =========================
class PlantDataset(Dataset):
    def __init__(self, root, transform=None):
        self.samples = []
        self.transform = transform
        for class_name in os.listdir(root):
            if class_name in class_mapping:
                class_path = os.path.join(root, class_name)
                for img in os.listdir(class_path):
                    if img.lower().endswith((".jpg", ".jpeg", ".png")):
                        self.samples.append((os.path.join(class_path, img), class_mapping[class_name]))

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform: image = self.transform(image)
        return image, label

# Load Data
plantdoc_root = "leaf_data_plantdoc/train"
full_pd_dataset = PlantDataset(plantdoc_root, transform=train_transform)
indices = list(range(len(full_pd_dataset)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

pd_train_subset = Subset(full_pd_dataset, train_idx)
pd_val_subset = Subset(PlantDataset(plantdoc_root, transform=val_transform), val_idx)
pv_dataset = PlantDataset("leaf_data/plantvillage dataset/color", transform=train_transform)

joint_train_dataset = ConcatDataset([pv_dataset, pd_train_subset])

# Sampler
targets = [label for dataset in joint_train_dataset.datasets for _, label in dataset]
targets = np.array(targets)
sample_weights = (1.0 / np.bincount(targets))[targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(joint_train_dataset, batch_size=16, sampler=sampler, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(pd_val_subset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

# =========================
# HYPER-FOCAL LOSS
# =========================
class FocalLoss(nn.Module):
    def __init__(self, gamma=3.0): # Gamma 3.0 = massive penalty for getting corn classes wrong
        super(FocalLoss, self).__init__()
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# =========================
# LOAD MODEL (DROP-PATH REGULARIZATION)
# =========================
model = timm.create_model(
    "vit_base_patch16_dinov3.lvd1689m", 
    pretrained=True, 
    img_size=384,
    drop_path_rate=0.1  # <--- The magic ViT anti-overfitting switch
)

for param in model.parameters(): param.requires_grad = False
for param in model.blocks[-6:].parameters(): param.requires_grad = True

model.head = nn.Sequential(nn.Dropout(0.5), nn.Linear(model.num_features, 10))
model = model.to(device)

# =========================
# OPTIMIZER (LOWER LR, HIGHER DECAY)
# =========================
criterion = FocalLoss(gamma=3.0)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-6,           # Slower, smoother learning rate
    weight_decay=0.05  # Strong penalty on large weights
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

# =========================
# TRAINING LOOP
# =========================
def train_model(model, train_loader, val_loader, epochs=40, patience=10):
    scaler = GradScaler("cuda")
    best_acc, early_stop = 0, 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0
        optimizer.zero_grad()

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            with autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            running_loss += loss.item()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                with autocast("cuda"): outputs = model(images)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (preds == labels).sum().item()

        val_acc = correct / total
        print(f"Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f} | Val Acc={val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_dino3_final.pth")
            early_stop = 0
        else:
            early_stop += 1

        if early_stop >= patience:
            print("Early stopping triggered.")
            break

        scheduler.step()
    print("Best Validation Accuracy:", best_acc)

train_model(model, train_loader, val_loader, epochs=40)

# =========================
# TEST EVALUATION WITH TTA
# =========================
pd_test_dataset = PlantDataset("test_dataset/plantdoc_test2", transform=val_transform)
test_loader = DataLoader(pd_test_dataset, batch_size=16, shuffle=False)

model = timm.create_model("vit_base_patch16_dinov3.lvd1689m", pretrained=False, img_size=384)
model.head = nn.Sequential(nn.Dropout(0.5), nn.Linear(model.num_features, 10))
model.load_state_dict(torch.load("best_dino3_final.pth", map_location=device))
model = model.to(device)
model.eval()

correct, total = 0, 0
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs_normal = model(images)
        outputs_flipped = model(torch.flip(images, dims=[3]))
        outputs_averaged = (outputs_normal + outputs_flipped) / 2.0
        
        _, preds = torch.max(outputs_averaged, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\n==============================")
print("FINAL TEST ACCURACY (DropPath + Hyper-Focal + TTA):", correct / total)
print("==============================")

class_names = [
    "Corn_Gray_leaf_spot", "Corn_leaf_blight", "Corn_rust_leaf", 
    "Tomato_Septoria_leaf_spot", "Tomato_leaf", "Apple_Scab_Leaf", 
    "Apple_leaf", "Apple_rust_leaf", "grape_leaf", "grape_leaf_black_rot"
]
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

Using device: cuda
Epoch 1: Loss=1.0369 | Val Acc=0.6203
Epoch 2: Loss=0.3851 | Val Acc=0.7219
Epoch 3: Loss=0.2379 | Val Acc=0.7807
Epoch 4: Loss=0.1573 | Val Acc=0.7861
Epoch 5: Loss=0.1192 | Val Acc=0.8235
Epoch 6: Loss=0.0950 | Val Acc=0.8128
Epoch 7: Loss=0.0763 | Val Acc=0.8182
Epoch 8: Loss=0.0696 | Val Acc=0.8663
Epoch 9: Loss=0.0658 | Val Acc=0.8556
Epoch 10: Loss=0.0547 | Val Acc=0.8610
Epoch 11: Loss=0.0514 | Val Acc=0.8717
Epoch 12: Loss=0.0436 | Val Acc=0.8770
Epoch 13: Loss=0.0460 | Val Acc=0.8824
Epoch 14: Loss=0.0412 | Val Acc=0.8770
Epoch 15: Loss=0.0367 | Val Acc=0.8824
Epoch 16: Loss=0.0339 | Val Acc=0.8930
Epoch 17: Loss=0.0354 | Val Acc=0.8877
Epoch 18: Loss=0.0330 | Val Acc=0.8930
Epoch 19: Loss=0.0332 | Val Acc=0.8824
Epoch 20: Loss=0.0306 | Val Acc=0.8984
Epoch 21: Loss=0.0297 | Val Acc=0.8877
Epoch 22: Loss=0.0263 | Val Acc=0.8930
Epoch 23: Loss=0.0285 | Val Acc=0.9037
Epoch 24: Loss=0.0258 | Val Acc=0.8930
Epoch 25: Loss=0.0261 | Val Acc=0.8984
Epoch 26: Loss=